In [1]:
import os
os.chdir('../')

In [40]:
import pandas as pd
import numpy as np
import pickle

from textwave.modules.extraction.preprocessing import DocumentProcessing
from textwave.modules.extraction.embedding import Embedding
from textwave.modules.retrieval.index.bruteforce import FaissBruteForce
from textwave.modules.retrieval.index.hnsw import FaissHNSW
from textwave.modules.retrieval.index.lsh import FaissLSH
from textwave.modules.retrieval.search import FaissSearch

print('DONE')

DONE


In [122]:
QUESTIONS_PATH = 'textwave/qa_resources/question.tsv'
CORPUS_PATH = 'textwave/storage/'
CHUNKING_STRATEGY = 'fixed-length' # 'fixed-length' or 'sentence'
CHUNKING_PARAMETERS = {
    "chunk_size": 150, 
    "overlap_size": 0,
    "num_sentences": 3,
}
INDEX_STRATEGY = "lsh"
INDEX_PARAMETERS = {
    'metric': 'cosine',
}
K_NEAREST_NEIGHBORS = 3

In [123]:
### Generate index ###

# Initialize objects
processing = DocumentProcessing()
embedding_model = Embedding()

# Preprocess and gather embeddings for corpus docs
metadata = []
embeddings = []
for doc in os.listdir(CORPUS_PATH):
    print(doc)

    # Process document into chunks
    chunk_size, overlap = CHUNKING_PARAMETERS['chunk_size'], CHUNKING_PARAMETERS['overlap_size']
    if CHUNKING_STRATEGY == 'fixed-length':
        chunks = processing.fixed_length_chunking(f"{CORPUS_PATH}/{doc}", chunk_size=chunk_size, overlap_size=overlap)
    elif CHUNKING_STRATEGY == 'sentence':
        chunks = processing.sentence_chunking(f"{CORPUS_PATH}/{doc}", num_sentences=CHUNKING_PARAMETERS['num_sentences'], overlap_size=overlap)
    else:
        raise ValueError('CHUNKING_STRATEGY must be one of: "fixed-length" or "sentence"')
    
    # Collect embeddings for each chunk
    for chunk in chunks:
        metadata.append(chunk)
        embeddings.append(embedding_model.encode(chunk))

# Store embeddings in a FAISS index
if INDEX_STRATEGY == 'bruteforce':
    index = FaissBruteForce(len(embeddings[0]), INDEX_PARAMETERS['metric'])
elif INDEX_STRATEGY == 'hnsw':
    index = FaissHNSW(dim=len(embeddings[0]))
elif INDEX_STRATEGY == 'lsh':
    index = FaissLSH(dim=len(embeddings[0]))
else:
    ValueError('INDEX_STRATEGY must be one of: "bruteforce", "hnsw", or "lsh"')

# Add embeddings and metadata to index
index.add_embeddings(embeddings, metadata)

# Index save path
index_path = f'faiss/{INDEX_STRATEGY}_{INDEX_PARAMETERS["metric"]}_{CHUNKING_STRATEGY}_{CHUNKING_PARAMETERS["chunk_size"]}_{CHUNKING_PARAMETERS["overlap_size"]}.pkl'
index.save(index_path)

index

S09_set4_a6.txt.clean
S09_set1_a3.txt.clean
S09_set3_a6.txt.clean
S08_set1_a5.txt.clean
S08_set2_a6.txt.clean
S09_set1_a10.txt.clean
S09_set5_a10.txt.clean
S08_set1_a8.txt.clean
S10_set6_a3.txt.clean
S10_set4_a6.txt.clean
S10_set3_a6.txt.clean
S10_set1_a3.txt.clean
S08_set4_a10.txt.clean
S09_set2_a10.txt.clean
S08_set4_a6.txt.clean
S08_set3_a6.txt.clean
S08_set1_a3.txt.clean
S10_set1_a8.txt.clean
S10_set6_a8.txt.clean
S08_set3_a10.txt.clean
S09_set1_a5.txt.clean
S09_set2_a6.txt.clean
S09_set5_a6.txt.clean
S10_set1_a5.txt.clean
S10_set2_a6.txt.clean
S10_set6_a5.txt.clean
S10_set5_a6.txt.clean
S09_set1_a8.txt.clean
S09_set4_a7.txt.clean
S09_set5_a1.txt.clean
S09_set1_a2.txt.clean
S09_set3_a7.txt.clean
S09_set2_a1.txt.clean
S08_set1_a4.txt.clean
S08_set3_a1.txt.clean
S08_set2_a7.txt.clean
S08_set4_a1.txt.clean
S10_set6_a2.txt.clean
S10_set4_a7.txt.clean
S10_set4_a10.txt.clean
S10_set5_a1.txt.clean
S10_set3_a7.txt.clean
S10_set1_a2.txt.clean
S10_set2_a1.txt.clean
S08_set1_a9.txt.clean
S10_

In [ ]:
# index = FaissBruteForce.load('faiss/bruteforce_cosine_fixed-length_200_0.pkl')

In [96]:
with open('analysis/question_embeddings.pkl', 'rb') as f:
    embeddings_map = pickle.load(f)

In [124]:
# For each question in each row -- get embeddings
# For full corpus: set chunk type, set overlap 
# Then generate full-corpus index (bruteforce for now) - save this!
# Generate FAISS serach object
# Get search results (DONT re-rank)
# For each search result chunk, its POSITIVE if it matches any answer from the corresponding answer file in the df

questions = pd.read_table(QUESTIONS_PATH)

results = {'p':0,'n':0,}
count = 1
filtered = questions[~questions['Question'].isna()]
filtered = filtered[~filtered['ArticleFile'].isna()]
for idx, row in filtered.iterrows():
    print(f'{count}/{len(filtered)}')

    # Embed query
    query = row['Question']
    query_vector = embeddings_map[query]

    # Search index for neighbors
    search = FaissSearch(index, metric=INDEX_PARAMETERS['metric'])
    _, _, meta_results = search.search(query_vector, k=K_NEAREST_NEIGHBORS)

    # Check answers
    answers_file = row['ArticleFile'] + '.txt.clean'
    p = 0
    n = 0
    with open(CORPUS_PATH + answers_file , 'r', errors='ignore') as file:
        text = file.read()
        for chunk in meta_results:
            if chunk in text:
                p += 1
                results['p'] += 1
            else:
                n += 1
                results['n'] += 1

    # Post results
    print(query)
    print(f'Positives: {p}, Negatives: {n}')
    print()
    count += 1


1/1032
Was Abraham Lincoln the sixteenth President of the United States?
Positives: 0, Negatives: 3

2/1032
Was Abraham Lincoln the sixteenth President of the United States?
Positives: 0, Negatives: 3

3/1032
Did Lincoln sign the National Banking Act of 1863?
Positives: 0, Negatives: 3

4/1032
Did Lincoln sign the National Banking Act of 1863?
Positives: 0, Negatives: 3

5/1032
Did his mother die of pneumonia?
Positives: 1, Negatives: 2

6/1032
Did his mother die of pneumonia?
Positives: 1, Negatives: 2

7/1032
How many long was Lincoln's formal education?
Positives: 0, Negatives: 3

8/1032
How many long was Lincoln's formal education?
Positives: 0, Negatives: 3

9/1032
When did Lincoln begin his political career?
Positives: 0, Negatives: 3

10/1032
When did Lincoln begin his political career?
Positives: 0, Negatives: 3

11/1032
What did The Legal Tender Act of 1862 establish?
Positives: 0, Negatives: 3

12/1032
What did The Legal Tender Act of 1862 establish?
Positives: 0, Negatives: 

In [125]:
print(results)
p = results['p']
n = results['n']

round(p / (p + n), 4)

{'p': 664, 'n': 2432}


0.2145

In [38]:
# Seperately embed each question - do this only once!
questions = pd.read_table(QUESTIONS_PATH)

filtered = questions[~questions['Question'].isna()]
count = 1
embedding_map = {}
for idx, row in filtered.iterrows():
    print(f'{count}/{len(filtered)}')

    # Embed query
    query = row['Question']
    query_vector = Embedding().encode(query)
    count += 1

    # Add embedding to map
    embedding_map[row['Question']] = query_vector
    
embedding_map

1/1034
2/1034
3/1034
4/1034
5/1034
6/1034
7/1034
8/1034
9/1034
10/1034
11/1034
12/1034
13/1034
14/1034
15/1034
16/1034
17/1034
18/1034
19/1034
20/1034
21/1034
22/1034
23/1034
24/1034
25/1034
26/1034
27/1034
28/1034
29/1034
30/1034
31/1034
32/1034
33/1034
34/1034
35/1034
36/1034
37/1034
38/1034
39/1034
40/1034
41/1034
42/1034
43/1034
44/1034
45/1034
46/1034
47/1034
48/1034
49/1034
50/1034
51/1034
52/1034
53/1034
54/1034
55/1034
56/1034
57/1034
58/1034
59/1034
60/1034
61/1034
62/1034
63/1034
64/1034
65/1034
66/1034
67/1034
68/1034
69/1034
70/1034
71/1034
72/1034
73/1034
74/1034
75/1034
76/1034
77/1034
78/1034
79/1034
80/1034
81/1034
82/1034
83/1034
84/1034
85/1034
86/1034
87/1034
88/1034
89/1034
90/1034
91/1034
92/1034
93/1034
94/1034
95/1034
96/1034
97/1034
98/1034
99/1034
100/1034
101/1034
102/1034
103/1034
104/1034
105/1034
106/1034
107/1034
108/1034
109/1034
110/1034
111/1034
112/1034
113/1034
114/1034
115/1034
116/1034
117/1034
118/1034
119/1034
120/1034
121/1034
122/1034
123/1034
1

{'Was Abraham Lincoln the sixteenth President of the United States?': array([-3.75676225e-03,  3.79637815e-02, -3.18204500e-02, -4.99767475e-02,
        -5.04510440e-02, -2.37986329e-03,  1.97595339e-02, -1.84855852e-02,
        -5.41222617e-02,  3.56299840e-02,  2.43018772e-02, -1.82908773e-02,
         5.11143096e-02, -6.17569126e-02, -3.32004428e-02,  1.73406694e-02,
         1.12314930e-03,  7.76945949e-02,  6.41778926e-04, -1.46901691e-02,
        -3.60674993e-03, -4.15751562e-02, -2.01918744e-02, -6.94234893e-02,
         5.59431277e-02,  7.23480657e-02, -4.02771123e-02, -8.36174414e-02,
        -5.32066040e-02, -1.84386205e-02,  2.58475039e-02, -1.52454555e-01,
         5.57588376e-02, -4.45946008e-02, -1.77140441e-02, -6.78597242e-02,
         7.41405413e-02,  5.17180562e-02,  4.51799743e-02, -6.72454908e-02,
        -2.62642801e-02,  2.88162865e-02,  3.12006902e-02,  2.04921756e-02,
        -3.74445668e-03, -3.40653621e-02, -4.51500155e-02, -3.45697775e-02,
         5.42752221

In [ ]:
# Save question embeddings
# with open('analysis/question_embeddings.pkl', 'wb') as f:
#     pickle.dump(embedding_map, f)